# 醫學影像 YOLO26 檢測（Kaggle）

1. 右側 **Settings**：Accelerator 選 **GPU**，**Internet** 打開。
2. 用 **Add Data** 加入程式（本 repo 的 `kaggle/` 資料夾）與你的醫學影像資料集。
3. 依序執行下面的格子。輸出在 `/kaggle/working/medical/`。

> ⚠️ 僅供研究與教學，不可作為臨床診斷依據。

In [ ]:
# 取得程式碼：
#   A) 已把本 repo 的 kaggle/ 資料夾上傳成 Kaggle Dataset 並用 Add Data 加入 → 自動找到
#   B) 否則從 GitHub clone（repo 需為公開）
import glob, os, subprocess

hits = [p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + "vi_core.py")]
if hits:
    CODE = os.path.dirname(hits[0])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/MorrisLaLa/visual-inspection.git", "/kaggle/working/visual-inspection"],
                   check=True)
    CODE = "/kaggle/working/visual-inspection/kaggle"
print("程式位置：", CODE)
!pip install -q -U ultralytics

## 示範：Ultralytics 內建 brain-tumor 資料集（自動下載）

In [ ]:
!python {CODE}/medical_yolo.py --epochs 50

In [ ]:
# 顯示報告與訓練圖表
from pathlib import Path
from IPython.display import Image, Markdown, display

OUT = Path("/kaggle/working/medical")
display(Markdown((OUT / "report.md").read_text()))
for f in ["train/results.png", "train/confusion_matrix_normalized.png", "val_test/BoxPR_curve.png"]:
    if (OUT / f).exists():
        display(Image(filename=str(OUT / f), width=900))

## 換成你的資料（把路徑改成 Add Data 後的實際位置，取消註解執行）

In [ ]:
# VinDr-CXR（DICOM + CSV）
# !python {CODE}/medical_yolo.py --csv /kaggle/input/<vindr>/train.csv --images /kaggle/input/<vindr>/train --merge-iou 0.4 --clahe --max-size 1024 --imgsz 1024

# RSNA Pneumonia（patientId, x, y, width, height, Target）
# !python {CODE}/medical_yolo.py --csv /kaggle/input/<rsna>/stage_2_train_labels.csv --images /kaggle/input/<rsna>/stage_2_train_images --image-col patientId --patient-col patientId --class-name pneumonia --max-size 1024 --imgsz 1024

# Kvasir-SEG 分割（影像 + 遮罩）
# !python {CODE}/medical_yolo.py --data /kaggle/input/<kvasir>/Kvasir-SEG/images --masks /kaggle/input/<kvasir>/Kvasir-SEG/masks --mask-classes polyp

# Chest X-Ray Pneumonia 分類（依病人編號分組重切）
# !python {CODE}/medical_yolo.py --data /kaggle/input/<chest-xray>/chest_xray --resplit --group-regex '^(person\d+|NORMAL2-IM-\d+|IM-\d+)'

## 閉環：下一輪
把 `review_queue.zip` 下載下來人工修正標註、併入資料集後，用上一輪的 `best.pt` 接續訓練並比較：

In [ ]:
# !python {CODE}/medical_yolo.py --data <新版資料> --model /kaggle/input/<上一輪輸出>/best.pt --baseline /kaggle/input/<上一輪輸出>/best.pt --unlabeled <未標註影像資料夾>